In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=kWapUAxrBo1YX0c8n5p0OrDoQ9zkIC&access_type=offline&code_challenge=7JQtrBwUK9Wf17PzCVgxNBtclp0tSSiJ_Co9Q0txWVA&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [2]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=9Z6XoayB2XXajYUtqZ1GtCzL976BRL&access_type=offline&code_challenge=vwsW1IdVSA7-SIcAvy6l542W4YVnNsvcq-KE5rwpmAk&code_challenge_method=S256


You are now logged in as [yt4@sanger.ac.uk].
Your current project is [open-targets-genetics-dev].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [1]:
import os
import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep

Loading BokehJS ...

In [ ]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/11 16:15:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
session.spark.sparkContext._conf.getAll()

[('spark.dynamicAllocation.initialExecutors', '2'),
 ('spark.driver.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false'),
 ('spark.shuffle.service.enabled', 'true'),
 ('spar

In [6]:
sl=StudyLocus.from_parquet(session,"gs://ot_orchestration/releases/25.02_freeze1/credible_set/")

In [ ]:
sl.df.filter(f.col("finemappingMethod")=="pics").groupBy("qualityControls").count().show(100)

+--------------------+------+
|     qualityControls| count|
+--------------------+------+
|[Variant not foun...|  5549|
|                  []| 24535|
|[LD block does no...|   315|
|[Study locus from...|  3029|
|[Study locus from...|  3620|
|[Study has qualit...| 20385|
|[LD block does no...|  6510|
|[Study locus from...|131146|
|[Variant not foun...|  1604|
+--------------------+------+



In [ ]:
sl1=sl.df.filter(f.col("studyLocusId")=="0154428c4fcde9d5ba7efb2610f11f04").cache()

In [ ]:
sl1.show(truncate=False)

+--------------------------------+---------+--------------+----------+--------+------+------------+-------+------+--------------+--------------+-------------------------------+-------------+-------------------+------------------------------------------------------------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+--------------------------------------------------------------------------------------------+-----+---------------------------------------------------------------+
|studyLocusId                    |studyType|variantId     |chromosome|position|region|studyId     |beta   |zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls                                                   |finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|locusStart|locusEnd|sampleSize|ldSet                                                         

In [ ]:
sl1.printSchema()

root
 |-- studyLocusId: string (nullable = true)
 |-- studyType: string (nullable = true)
 |-- variantId: string (nullable = true)
 |-- chromosome: string (nullable = true)
 |-- position: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- studyId: string (nullable = true)
 |-- beta: double (nullable = true)
 |-- zScore: double (nullable = true)
 |-- pValueMantissa: float (nullable = true)
 |-- pValueExponent: integer (nullable = true)
 |-- effectAlleleFrequencyFromSource: float (nullable = true)
 |-- standardError: double (nullable = true)
 |-- subStudyDescription: string (nullable = true)
 |-- qualityControls: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- finemappingMethod: string (nullable = true)
 |-- credibleSetIndex: integer (nullable = true)
 |-- credibleSetlog10BF: double (nullable = true)
 |-- purityMeanR2: double (nullable = true)
 |-- purityMinR2: double (nullable = true)
 |-- locusStart: integer (nullable = true)
 |-- locusEnd

In [ ]:
sl1.withColumn(
    "ldSet",
    f.expr(
        """
        transform(ldSet, x -> 
            IF(x.tagVariantId == variantId, 
                named_struct('tagVariantId', x.tagVariantId, 'r2Overall', 1.0), 
                x
            )
        )
        """
    )
).show(truncate=False)

+--------------------------------+---------+--------------+----------+--------+------+------------+-------+------+--------------+--------------+-------------------------------+-------------+-------------------+------------------------------------------------------------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+--------------------------------------------------------------------------------------------+-----+---------------------------------------------------------------+
|studyLocusId                    |studyType|variantId     |chromosome|position|region|studyId     |beta   |zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls                                                   |finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|locusStart|locusEnd|sampleSize|ldSet                                                         

In [ ]:
sl1.withColumn("locus_size", f.size("locus")).filter(f.col("locus_size")<=0).show(truncate=False)

+--------------------------------+---------+--------------+----------+--------+------+------------+-------+------+--------------+--------------+-------------------------------+-------------+-------------------+------------------------------------------------------------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+--------------------------------------------------------------------------------------------+-----+---------------------------------------------------------------+----------+
|studyLocusId                    |studyType|variantId     |chromosome|position|region|studyId     |beta   |zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls                                                   |finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|locusStart|locusEnd|sampleSize|ldSet                                              

In [ ]:
sl.df.groupBy("finemappingMethod").count().show()

+-----------------+-------+
|finemappingMethod|  count|
+-----------------+-------+
|            SuSie|2044538|
|        SuSiE-inf| 291867|
+-----------------+-------+



In [ ]:
sl.df.show()

+--------------------+---------+--------------------+----------+---------+--------------------+--------------------+--------------------+------------------+--------------+--------------+-------------------------------+-------------+-------------------+--------------------+-----------------+----------------+------------------+------------------+------------------+----------+--------+----------+--------------------+--------------------+--------------------+
|        studyLocusId|studyType|           variantId|chromosome| position|              region|             studyId|                beta|            zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|     qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|      purityMeanR2|       purityMinR2|locusStart|locusEnd|sampleSize|               ldSet|               locus|          confidence|
+--------------------+---------+--------------------+----------+---------+------

In [ ]:
sl1=sl.df.withColumn("locus_size", f.size("locus")).filter(f.col("locus_size")<=0).cache()
sl1.count()

10445

In [ ]:
sl1.groupBy("finemappingMethod").count().show()

+-----------------+-----+
|finemappingMethod|count|
+-----------------+-----+
|             pics|10445|
+-----------------+-----+



In [ ]:
sl2=sl1.withColumn(
                    "ldSet",
                    f.expr(
                        """
                        transform(ldSet, x ->
                            IF(x.tagVariantId == variantId,
                                named_struct('tagVariantId', x.tagVariantId, 'r2Overall', 1.0),
                                x
                            )
                        )
                        """
                    ),
                )

In [ ]:
sl1.count()

10445

In [ ]:

exploded_df = sl2.withColumn("ldSet", f.explode("ldSet"))
filtered_df = exploded_df.filter(f.col("variantId") == f.col("ldSet.tagVariantId"))
result_df = filtered_df.select("variantId", f.col("ldSet.r2Overall").alias("r2Overall"))


In [ ]:
result_df.describe().show()

+-------+----------------+---------+
|summary|       variantId|r2Overall|
+-------+----------------+---------+
|  count|           10445|    10445|
|   mean|            null|      1.0|
| stddev|            null|      0.0|
|    min|10_100294413_C_T|      1.0|
|    max|  X_97757141_A_C|      1.0|
+-------+----------------+---------+



In [ ]:
sl2.show(1,truncate=False)

+--------------------------------+---------+---------------+----------+--------+------+------------+-----+------+--------------+--------------+-------------------------------+-------------+-------------------+------------------------------------------------------------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
sl3=StudyLocus(_df=sl2.drop("locus_size"),_schema=sl.schema)
from gentropy.config import WindowBasedClumpingStepConfig
from gentropy.dataset.study_locus import CredibleInterval, StudyLocus
from gentropy.method.pics import PICS
PICS.finemap(sl3).filter_credible_set(credible_interval=CredibleInterval.IS99).df.withColumn("locus_size2", f.size("locus")).groupBy("locus_size2").count().show()  

+-----------+-----+
|locus_size2|count|
+-----------+-----+
|          1|10445|
+-----------+-----+



In [ ]:
sl.df.count()

2534545

In [ ]:
sl2.select("qualityControls").show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------+
|qualityControls                                                                                                                        |
+---------------------------------------------------------------------------------------------------------------------------------------+
|[LD block does not contain variants at the required R^2 threshold]                                                                     |
|[LD block does not contain variants at the required R^2 threshold]                                                                     |
|[LD block does not contain variants at the required R^2 threshold]                                                                     |
|[LD block does not contain variants at the required R^2 threshold]                                                                     |
|[Study locus from curated top hit

In [ ]:
filtered_df = sl.df.filter(
    f.array_contains(f.col("qualityControls"), "LD block does not contain variants at the required R^2 threshold")
)
filtered_df.count()

10445

In [ ]:
sl=sl.df.filter(f.col("studyType")=="gwas").cache()
sl.count()

277373

In [ ]:
sl.groupBy("finemappingMethod").count().show()

+-----------------+------+
|finemappingMethod| count|
+-----------------+------+
|            SuSie| 16055|
|        SuSiE-inf|261318|
+-----------------+------+



In [ ]:
x=session.spark.read.parquet("temp_fiels/nan_locus2")
from pyspark.sql.types import StringType
# Assuming x is your DataFrame
x = x.withColumn("studyLocusId", f.col("studyLocusId").cast(StringType()))

df2=StudyLocus(
            _df=x,
            _schema=StudyLocus.get_schema(),
        )

df2=StudyLocus.from_parquet(session, "temp_fiels/nan_locus3")
df2=df2.df
#df2.show()

In [ ]:
si=StudyIndex.from_parquet(session,"gs://genetics_etl_python_playground/releases/24.06/study_index/gwas_catalog")

In [ ]:
session=session
study_locus_row=df2.collect()[3]
study_index=si
max_causal_snps=10
cs_lbf_thr=2
sum_pips=0.99
susie_est_tausq=False
run_carma=False
carma_tau=0.04
run_sumstat_imputation=False
carma_time_limit=6000
imputed_r2_threshold=0.8
ld_score_threshold=5
purity_mean_r2_threshold=0
purity_min_r2_threshold=0.25
lead_pval_threshold=1e-5
ld_min_r2=0.8

In [ ]:
result_logging = SusieFineMapperStep.susie_finemapper_one_sl_row_gathered_boundaries(
    session=session,
    study_locus_row=study_locus_row,
    study_index=study_index,
    max_causal_snps=max_causal_snps,
    purity_mean_r2_threshold=purity_mean_r2_threshold,
    purity_min_r2_threshold=purity_min_r2_threshold,
    cs_lbf_thr=cs_lbf_thr,
    sum_pips=sum_pips,
    lead_pval_threshold=1e-5,
    susie_est_tausq=susie_est_tausq,
    run_carma=False,
    run_sumstat_imputation=run_sumstat_imputation,
    carma_tau=0.04,
    carma_time_limit=carma_time_limit,
    imputed_r2_threshold=imputed_r2_threshold,
    ld_score_threshold=ld_score_threshold,
    ld_min_r2=ld_min_r2,
)

In [ ]:
result_logging["log"]

,studyId,region,N_gwas_before_dedupl,N_gwas,N_ld,N_overlap,N_outliers,N_imputed,N_final_to_fm,elapsed_time,number_of_CS,error
0,GCST90025970,3:46513902-48013902,846,846,846,846,0,0,846,38.45847,5,


In [ ]:
SusieFineMapperStep._empty_log_mg(studyId="ss",region="sss",error_mg="ssffffs",path_out="temp_fiels/nan_locus6")

In [ ]:
result_logging["log"].to_csv(,index=False,sep="\t")

In [ ]:
log_output=""

In [ ]:
log_output!=""

False

In [ ]:
result_logging["study_locus"] is None

False

In [ ]:
result_logging["study_locus"].df.show(truncate=False)

+----------------+--------------------------------+------------+-------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+----------+--------+-----------------+------------------+------------------+------------------+------------------+--------------+--------------+----------+--------+---------------------+
|credibleSetIndex|studyLocusId                    |studyId     |region             |locus                                                                                                                                                                                                                                                                                                                

In [ ]:
StudyLocus(_df=result_logging["study_locus"].df,_schema=StudyLocus.get_schema())

StudyLocus(_df=DataFrame[credibleSetIndex: int, studyLocusId: string, studyId: string, region: string, locus: array<struct<variantId:string,posteriorProbability:double,logBF:double,beta:double>>, variantId: string, chromosome: string, position: int, finemappingMethod: string, credibleSetlog10BF: double, purityMeanR2: double, purityMinR2: double, zScore: double, pValueMantissa: float, pValueExponent: int, locusStart: int, locusEnd: int, beta: double], _schema=StructType([StructField('studyLocusId', StringType(), False), StructField('studyType', StringType(), True), StructField('variantId', StringType(), False), StructField('chromosome', StringType(), True), StructField('position', IntegerType(), True), StructField('region', StringType(), True), StructField('studyId', StringType(), False), StructField('beta', DoubleType(), True), StructField('zScore', DoubleType(), True), StructField('pValueMantissa', FloatType(), True), StructField('pValueExponent', IntegerType(), True), StructField('ef